# Clustering

Until now every model in the course had a target: a price, a churn flag, a busy listing. This notebook starts the part of machine learning that works without one and covers the first block of Session 11. It explains what unsupervised learning can and cannot deliver, why distances and scaling decide the result, how k-means works (computed by hand on six numbers) and how to choose the number of clusters with the elbow method and the silhouette coefficient. The running example of these sections is the Telco customer data of Sessions 6 and 8, now used without the churn label. The second half adds two methods with other assumptions, **hierarchical clustering** and **DBSCAN**, on the Berlin listings of Inside Airbnb, the second dataset of the course, and ends with the question that matters most in practice: is a clustering any good?

The code cells in this notebook build on each other; run them in order. The first cell moves to the repository root, where the case-study paths start.

```mermaid
mindmap
  root((Unsupervised learning))
    Clustering
      k-means
      hierarchical
      DBSCAN
    Dimensionality reduction
      PCA
      t-SNE and UMAP
    Anomaly detection
      Isolation Forest
      local outlier factor
```

In [1]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

## Supervised and unsupervised learning

### Concept

In **supervised learning** each training row has inputs X and a known answer y (the **target** or **label**). The model learns a function from X to y, and we can measure how often it is right on new data. In **unsupervised learning** there is only X. The task is to describe structure in X: groups of similar rows (**clustering**), a few directions that summarise many columns (**dimensionality reduction**), or rows that do not fit the rest (**anomaly detection**).

A small example. A table of 7,032 Telco customers has the columns tenure (months as a customer), monthly charges and total charges, plus the column Churn (Yes/No).

- Supervised question: "Will this customer churn?" The model uses Churn as y and can be scored with accuracy or ROC AUC.
- Unsupervised question: "Which kinds of customers do we have?" The method sees only the three numeric columns. Churn can be used *afterwards* to describe the groups, but not to build them.

The difficulty is evaluation. Without y there is no "correct" grouping to compare with. Two analysts can produce two different segmentations, and both can be reasonable. Evaluation therefore uses three kinds of evidence:

1. **Internal criteria**: numbers computed from X and the clusters alone, such as the within-cluster sum of squares or the silhouette coefficient (below).
2. **Stability**: does the result stay the same on a different sample or with a different random start? (last section of this notebook)
3. **Usefulness**: can a person interpret the groups, and do they help a decision or a later model? (last section of this notebook, and the lecture of block 2)

```mermaid
flowchart LR
    Q{"Is there a target y<br/>in the training data?"} -->|yes| S["Supervised:<br/>regression, classification"]
    Q -->|no| U["Unsupervised:<br/>clusters, components, anomalies"]
    S --> SE["Evaluate against y<br/>on a test set"]
    U --> UE["Evaluate with internal criteria,<br/>stability and usefulness"]
```

### Why it matters

Most data in organisations have no labels: customer records, sensor readings, documents. Labelling costs time and money. Unsupervised methods give a first structure (segments, a map of the data, a list of suspicious rows) that people can then inspect, name and act on. They are also used inside supervised pipelines, for example as extra features (lecture of block 2).

### How it works in Python

In [2]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

url = ("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
       "master/data/Telco-Customer-Churn.csv")
telco = pd.read_csv(url)
telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce")  # 11 blanks -> NaN
telco = telco.dropna(subset=["TotalCharges"])

X = telco[["tenure", "MonthlyCharges", "TotalCharges"]]   # the only input of the clustering
churn = telco["Churn"]                                     # kept aside to describe the clusters later
print(X.shape)                                             # (7032, 3)
print(X.describe().loc[["mean", "std"]].round(1))
#       tenure  MonthlyCharges  TotalCharges
# mean    32.4            64.8        2283.3
# std     24.5            30.1        2266.8

(7032, 3)
      tenure  MonthlyCharges  TotalCharges
mean    32.4            64.8        2283.3
std     24.5            30.1        2266.8


Note the API difference to supervised learning: `KMeans().fit(X)` takes no y.

### In practice

- **Cancer subtypes.** Perou et al. (2000, *Nature*) clustered gene-expression profiles of breast tumours without using outcomes and found subtypes (luminal, basal-like, HER2-enriched) that were later shown to differ in survival. The clinical value was established afterwards, with labelled data.
- **Geodemographic classification.** The UK Office for National Statistics publishes the Output Area Classification, which groups small areas into "supergroups" by census variables with k-means. Local authorities use it to target services.
- **Population structure.** Novembre et al. (2008, *Nature*) showed that the first two principal components of genetic data from Europeans reproduce the map of Europe, with no geographic labels in the input.

> **Warning.** **A clustering always returns clusters.** k-means with k = 4 returns four groups even for uniform random noise. The algorithm does not tell you whether the groups exist; your evaluation must.

> **Caution.** **Do not cluster on the target.** If churn is part of X, the "segments" partly reproduce churn and look more useful than they are. Keep the label out of the clustering and use it only to describe the result.

## Distance, similarity and scaling

### Concept

Clustering groups rows that are "close". Closeness is measured by a **distance** (small = similar) or a **similarity** (large = similar).

- **Euclidean distance** between rows a and b with p columns: d(a, b) = √((a₁ − b₁)² + … + (aₚ − bₚ)²). It is the straight-line distance and the default of k-means.
- **Manhattan distance**: |a₁ − b₁| + … + |aₚ − bₚ|, the distance along a city grid. It is less dominated by one large difference.
- **Cosine similarity**: cos θ = (a · b) / (‖a‖ ‖b‖), the angle between two vectors, ignoring their length. It is used for text vectors (Sessions 13 and 14), where a long and a short document on the same topic should be similar.

Worked example with two columns, tenure (months) and total charges (dollars):

| Customer | tenure | total charges |
|---|---|---|
| A | 2 | 150 |
| B | 70 | 150 |
| C | 2 | 1,150 |

On the raw scale d(A, B) = √(68² + 0²) = 68 and d(A, C) = √(0² + 1000²) = 1,000. A is "closer" to B, although B has been a customer 35 times longer. The dollar column dominates because its numbers are larger. After **standardising** each column to mean 0 and standard deviation 1 (dividing tenure by its SD of 24.5 and total charges by 2,266.8), the distances become 68/24.5 = 2.78 and 1000/2266.8 = 0.44: now A is closer to C.

```mermaid
flowchart TD
    A["Choose the columns<br/>that define 'similar'"] --> B{"Same unit<br/>and range?"}
    B -->|no| C["Standardise<br/>(StandardScaler)"]
    B -->|yes| D["Keep the raw scale"]
    C --> E{"Strong skew<br/>or outliers?"}
    D --> E
    E -->|yes| F["Log transform first<br/>or RobustScaler"]
    E -->|no| G["Cluster"]
    F --> G
```

### Why it matters

Every distance-based method (k-means, hierarchical clustering, DBSCAN, k-nearest neighbours, PCA) inherits the scale of the columns. Without scaling, the column with the largest numbers decides the clusters, regardless of its importance. The choice of columns matters just as much: adding ten columns about phone services makes "similar" mean "similar in phone services".

### How it works in Python

In [3]:
# k-means on the raw columns: total charges (range 19-8,685) dominates
raw = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X)
print(X.groupby(raw.labels_).mean().round(0))
#    tenure  MonthlyCharges  TotalCharges
# 0    18.0            50.0         686.0
# 1    64.0            98.0        6293.0
# 2    44.0            78.0        3273.0     <- three bands of total charges

# k-means on standardised columns: all three columns count equally
Z = StandardScaler().fit_transform(X)
scaled = KMeans(n_clusters=3, n_init=10, random_state=0).fit(Z)
print(pd.Series(scaled.labels_).value_counts().sort_index().to_dict())
# {0: 2200, 1: 2147, 2: 2685}

   tenure  MonthlyCharges  TotalCharges
0    18.0            50.0         686.0
1    64.0            98.0        6293.0
2    44.0            78.0        3273.0


{0: 2200, 1: 2147, 2: 2685}


### In practice

- **Customer analytics.** Retail and telecom segmentations typically standardise or log-transform spending variables before clustering, because spending is right-skewed.
- **Text and recommendation.** Search engines and recommender systems compare documents or users with cosine similarity, so that the length of a document or the activity of a user does not dominate.
- **Gene expression.** Eisen et al. (1998, *PNAS*) clustered genes with a correlation-based similarity, which compares the shape of expression profiles rather than their level.

> **Warning.** **Scaling is a modelling decision, not a formality.** Standardising gives every column the same weight. If one column should matter more, weight it explicitly, and write the choice down.

> **Tip.** One-hot encoded categories (0/1 columns) and standardised numeric columns are on different scales too. For mixed data, try a small number of numeric columns first, or look up Gower distance and the k-prototypes algorithm, which are designed for mixed data.

## k-means computed by hand

### Concept

**k-means** splits n rows into k clusters. Each cluster has a **centre** (the **centroid**, the mean of its rows). The algorithm (Lloyd's algorithm) repeats two steps:

1. **Assign** every row to the nearest centre.
2. **Update** every centre to the mean of the rows assigned to it.

It stops when the assignments no longer change. The quantity that k-means makes smaller in each step is the **inertia** or **within-cluster sum of squares** (WCSS): the sum of squared distances of the rows to their own centre.

Worked example in one dimension: the six values 1, 2, 3, 8, 9, 15 with k = 2 and the (poor) starting centres c₁ = 1 and c₂ = 2.

| Iteration | Assignment | New centres |
|---|---|---|
| 1 | {1} to c₁; {2, 3, 8, 9, 15} to c₂ | c₁ = 1, c₂ = (2 + 3 + 8 + 9 + 15) / 5 = 7.4 |
| 2 | 2 is now closer to 1 than to 7.4, and 3 too: {1, 2, 3} and {8, 9, 15} | c₁ = 2, c₂ = 32 / 3 ≈ 10.67 |
| 3 | 8 is 6 from c₁ and 2.67 from c₂: assignments unchanged | stop |

Inertia at the end: (1 − 2)² + (2 − 2)² + (3 − 2)² = 2 for the first cluster and (8 − 10.67)² + (9 − 10.67)² + (15 − 10.67)² ≈ 28.67 for the second, 30.67 in total.

![Four panels of k-means on three groups of points: the centres start in the wrong places and move to the group means within three iterations while the inertia falls from 1275 to 212](../figures/kmeans_iterations.png)

```mermaid
stateDiagram-v2
    [*] --> Initialise: choose k starting centres
    Initialise --> Assign
    Assign --> Update: each row to nearest centre
    Update --> Assign: centres move to cluster means
    Update --> [*]: assignments unchanged
```

### Why it matters

Computing the steps once by hand shows what k-means assumes. It uses means, so outliers pull centres. It uses Euclidean distance to one centre, so it prefers round clusters of similar size. And it depends on the start: Lloyd's algorithm only finds a **local** minimum of the inertia. scikit-learn therefore uses a careful start (**k-means++**, which spreads the first centres apart) and repeats the whole algorithm `n_init` times, keeping the best run.

### How it works in Python

In [4]:
x = np.array([1, 2, 3, 8, 9, 15], dtype=float)
centres = np.array([1.0, 2.0])                       # deliberately poor start
for it in range(1, 10):
    labels = np.argmin(np.abs(x[:, None] - centres), axis=1)    # step 1: assign
    new = np.array([x[labels == k].mean() for k in range(2)])   # step 2: update
    print(it, labels, new.round(2))
    if np.allclose(new, centres):
        break
    centres = new
# 1 [0 1 1 1 1 1] [1.  7.4]
# 2 [0 0 0 1 1 1] [ 2.   10.67]
# 3 [0 0 0 1 1 1] [ 2.   10.67]
inertia = ((x - centres[labels]) ** 2).sum()
print(round(inertia, 2))                              # 30.67

km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(x.reshape(-1, 1))
print(km.cluster_centers_.ravel().round(2), round(km.inertia_, 2))   # [10.67  2.  ] 30.67

1 [0 1 1 1 1 1] [1.  7.4]
2 [0 0 0 1 1 1] [ 2.   10.67]
3 [0 0 0 1 1 1] [ 2.   10.67]
30.67
[10.67  2.  ] 30.67


The order of the two centres in scikit-learn's output may differ; cluster numbers are arbitrary names.

### In practice

- **Colour quantisation.** Image software reduces an image to, for example, 16 colours by clustering the pixel colours with k-means and replacing each pixel by its centroid ([extras/01-pdsh-kmeans.ipynb](extras/01-pdsh-kmeans.ipynb) shows this).
- **Vector quantisation in signal coding.** The Linde–Buzo–Gray algorithm (1980), a k-means variant, built codebooks for speech compression.
- **Geodemographics.** The ONS Output Area Classification mentioned above is a k-means clustering of standardised census variables.

> **Warning.** **Cluster numbers are arbitrary.** Running k-means again can swap the labels 0, 1 and 2. Never hard-code "cluster 2 is the high-value segment"; identify clusters by their profile.

> **Caution.** **k-means assumes round, similar-sized clusters.** On elongated, nested or very unequal groups it cuts in the wrong places. [Extras notebook 03](extras/03-kmeans-assumptions.ipynb) (scikit-learn gallery) shows four such failures; DBSCAN and hierarchical clustering (later in this notebook) handle some of them.

## Choosing the number of clusters: elbow method and silhouette

### Concept

k must be chosen before k-means runs. Two common aids:

- **Elbow method.** Fit k-means for k = 1, 2, …, 10 and plot the inertia. It always decreases with k (with k = n every row is its own cluster and the inertia is 0). Look for the **elbow**, the k after which adding a cluster brings little reduction.
- **Silhouette coefficient.** For a row i, let a(i) be the mean distance to the other rows of its own cluster and b(i) the mean distance to the rows of the nearest other cluster. Then s(i) = (b(i) − a(i)) / max(a(i), b(i)). Values near 1 mean the row sits well inside its cluster; near 0 it lies between two clusters; below 0 it is probably in the wrong cluster. The **mean silhouette** over all rows summarises a clustering, and the k with the highest mean is a candidate.

By hand for the value 1 in the example above: a = (|1 − 2| + |1 − 3|)/2 = 1.5; b = (7 + 8 + 14)/3 ≈ 9.67; s = (9.67 − 1.5)/9.67 ≈ 0.84.

![Two line charts for data with four true groups: the inertia drops steeply until k = 4 and then flattens (the elbow), and the mean silhouette peaks at 0.81 for k = 4](../figures/elbow_silhouette.png)

### Why it matters

The choice of k changes the story told to the business: three segments or six. Both aids give evidence, not an answer. On clean toy data the elbow and the silhouette agree; on real data the elbow is often smooth and the silhouette flat. Then k is chosen by interpretability and purpose (how many segments can the marketing team serve?), checked with stability (last section of this notebook).

### How it works in Python

In [5]:
rows = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Z)
    rows.append({"k": k, "inertia": round(km.inertia_), "silhouette": round(silhouette_score(Z, km.labels_), 3)})
print(pd.DataFrame(rows).to_string(index=False))
#  k  inertia  silhouette
#  2     9701       0.480
#  3     6177       0.452
#  4     4145       0.472
#  5     3108       0.444
#  6     2560       0.438
#  7     2177       0.433
#  8     1883       0.431

 k  inertia  silhouette
 2     9701       0.480
 3     6177       0.452
 4     4145       0.472
 5     3108       0.444
 6     2560       0.438
 7     2177       0.433
 8     1883       0.431


No sharp elbow, and the silhouette varies little between 2 and 4. Three segments are a reasonable, interpretable choice. Describe them with their means and with the churn rate, which was not used for the clustering:

In [6]:
telco["segment"] = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(Z)
profile = telco.groupby("segment").agg(
    customers=("tenure", "size"),
    tenure=("tenure", "mean"),
    monthly=("MonthlyCharges", "mean"),
    churn_rate=("Churn", lambda s: (s == "Yes").mean()),
    month_to_month=("Contract", lambda s: (s == "Month-to-month").mean()),
)
print(profile.round(2))
#          customers  tenure  monthly  churn_rate  month_to_month
# segment
# 0             2200   58.56    89.70        0.15            0.26
# 1             2147   29.59    26.58        0.12            0.44
# 2             2685   13.27    74.96        0.47            0.87

         customers  tenure  monthly  churn_rate  month_to_month
segment                                                        
0             2200   58.56    89.70        0.15            0.26
1             2147   29.59    26.58        0.12            0.44
2             2685   13.27    74.96        0.47            0.87


A possible reading: segment 0 are long-standing customers with high bills, segment 1 low-cost customers, segment 2 new customers with high bills, mostly on monthly contracts; almost half of them churned. Names like these are interpretations that the business must confirm.

### In practice

- **Marketing segmentations** are usually settled at a k that the organisation can act on (for example four to eight segments with a named owner each), using statistics like these as a check.
- **The silhouette plot** of Rousseeuw (1987), a bar per row sorted within clusters, is a standard diagnostic; [extras/02-kmeans-silhouette-analysis.ipynb](extras/02-kmeans-silhouette-analysis.ipynb) (scikit-learn gallery) reproduces it for k = 2 to 6.
- **The gap statistic** (Tibshirani, Walther and Hastie, 2001) compares the inertia with that of uniform random data and is implemented in several statistics packages.

> **Warning.** **The silhouette favours round, well-separated clusters.** It can prefer a wrong k when the true groups are elongated or nested, and it is computed with the same distance as k-means. Treat it as one piece of evidence.

> **Tip.** `silhouette_score` computes all pairwise distances. On more than about 20,000 rows pass `sample_size=10_000, random_state=0` to keep it fast.

*Worked example:* [02-case-study-telco-segments.ipynb](02-case-study-telco-segments.ipynb) segments the Telco customers with k-means, checks the stability of the segments and describes them.

## From customers to listings

k-means needs the number of clusters in advance and assumes round groups. The next sections add two methods with other assumptions: **hierarchical clustering**, which builds a whole tree of nested clusters and draws it as a dendrogram, and **DBSCAN**, which finds dense regions of any shape and leaves sparse points as noise. The last section covers the silhouette coefficient for comparing methods, stability under resampling and cluster profiles for interpretation.

The running example are the Berlin listings of Inside Airbnb (snapshot of 26 June 2026; prepare the data with `uv run python case-study/prepare_airbnb.py`). The practical question: **what kinds of Airbnb offers exist in Berlin?** A city office that enforces the rules on short-term rentals, a tourism board or a new host would like a handful of types rather than 12,776 single listings. We use the 8,439 listings with a price and describe each by six numbers: the nightly price, the number of guests, the minimum number of nights, the number of nights bookable in the next year, the reviews of the last twelve months (a sign of recent activity) and the distance to Alexanderplatz. Price, minimum nights and reviews are strongly skewed, so they enter on the log scale (Session 4); then every column is standardised (as for the Telco customers above). Room type, district and host are left out of the clustering and used afterwards to describe the clusters.

```mermaid
flowchart TD
    Q1{"Do you know roughly<br/>how many clusters?"} -->|yes| Q2{"Round clusters of<br/>similar size?"}
    Q1 -->|no| Q3{"Expect noise points or<br/>clusters of odd shape?"}
    Q2 -->|yes| KM["k-means"]
    Q2 -->|no| HC["Hierarchical clustering<br/>(Ward or average linkage)"]
    Q3 -->|yes| DB["DBSCAN or HDBSCAN"]
    Q3 -->|no| HC
    KM --> EV["Evaluate: silhouette,<br/>stability, profiles"]
    HC --> EV
    DB --> EV
```

In [7]:
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import fcluster, linkage
from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 12)
listings = pd.read_parquet("case-study/data/airbnb/listings.parquet")
d = listings.dropna(subset=["price", "minimum_nights"]).reset_index(drop=True)   # 34 % have no price
lat0, lon0 = 52.5219, 13.4132                                   # Alexanderplatz
d["north_km"] = (d["latitude"] - lat0) * 111.2                  # degrees -> km (good enough within a city)
d["east_km"] = (d["longitude"] - lon0) * 111.2 * np.cos(np.radians(lat0))
d["km_centre"] = np.hypot(d["north_km"], d["east_km"])

features = pd.DataFrame({
    "log_price": np.log(d["price"]),                  # skewed: log first (Session 4)
    "guests": d["accommodates"],
    "log_min_nights": np.log(d["minimum_nights"]),
    "availability_365": d["availability_365"],        # nights bookable in the next year
    "log_reviews_ltm": np.log1p(d["number_of_reviews_ltm"]),   # reviews in the last 12 months
    "km_centre": d["km_centre"],
})
Z = StandardScaler().fit_transform(features)
print(Z.shape)                                        # (8439, 6)

(8439, 6)


> **Note.** **Ethics.** Inside Airbnb scrapes public listing pages. Describe clusters in aggregate (sizes, medians, shares) and never single out or name hosts.

## Hierarchical clustering and dendrograms

### Concept

**Agglomerative hierarchical clustering** starts with every row as its own cluster and repeatedly merges the two closest clusters until one cluster is left. The result is a tree. A **dendrogram** draws it: the leaves are the rows, each merge is a horizontal bar, and its height is the distance at which the two clusters were joined. Cutting the tree with a horizontal line at some height gives a flat clustering; the number of vertical lines the cut crosses is the number of clusters.

How far apart are two *clusters*? The **linkage** decides:

- **single**: the distance of the closest pair of rows (one from each cluster); can form long chains;
- **complete**: the distance of the farthest pair; gives compact clusters;
- **average**: the mean of all pairwise distances;
- **Ward**: the merge that increases the within-cluster sum of squares least; similar in spirit to k-means and the usual default.

Worked example: the four values 1, 2, 5, 11.

| Step | Single linkage | Complete linkage |
|---|---|---|
| 1 | merge {1} and {2} at 1 | merge {1} and {2} at 1 |
| 2 | d({1, 2}, 5) = 3 (from 2), d(5, 11) = 6: merge {1, 2, 5} at 3 | d({1, 2}, 5) = 4 (from 1), d(5, 11) = 6: merge {1, 2, 5} at 4 |
| 3 | d({1, 2, 5}, 11) = 6: merge all at 6 | d({1, 2, 5}, 11) = 10: merge all at 10 |

The trees have the same shape here but different heights. Cutting the complete-linkage tree between 4 and 10 gives two clusters, {1, 2, 5} and {11}.

![Left: 15 points in three coloured groups. Right: their Ward dendrogram; a dashed horizontal cut at height 4.7 crosses three branches, giving the same three groups](../figures/dendrogram.png)

### Why it matters

The dendrogram shows structure at every level at once: a few large groups that split into subgroups. One does not have to fix k in advance; long vertical branches (a large gap between merge heights) suggest a natural number of clusters. Hierarchies are also meaningful in themselves: product categories, biological taxonomies, organisational units, and Berlin itself (12 districts, each split into the neighbourhoods of the listings).

### How it works in Python

In [8]:
# the worked example
tiny = np.array([[1.0], [2.0], [5.0], [11.0]])
print(linkage(tiny, method="single")[:, 2])      # [1. 3. 6.]   merge heights
print(linkage(tiny, method="complete")[:, 2])    # [ 1.  4. 10.]

# the listings: Ward linkage on a random sample of 2,000 (memory grows with n squared)
rng = np.random.default_rng(0)
idx = rng.choice(len(Z), size=2000, replace=False)
Zs = Z[idx]
Z_ward = linkage(Zs, method="ward")
print(Z_ward[-5:, 2].round(1))                                  # [31.9 40.7 42.5 47.8 86.6]  last merge heights
ward_labels = fcluster(Z_ward, t=5, criterion="maxclust")       # labels 1..5
print(pd.Series(ward_labels).value_counts().sort_index().to_dict())
# {1: 377, 2: 713, 3: 179, 4: 141, 5: 590}

# the same with scikit-learn (labels 0..4, same partition)
agg = AgglomerativeClustering(n_clusters=5, linkage="ward").fit(Zs)
print(adjusted_rand_score(ward_labels, agg.labels_))            # 1.0

# draw the top of the tree (scipy.cluster.hierarchy.dendrogram):
# dendrogram(Z_ward, truncate_mode="lastp", p=20)

[1. 3. 6.]
[ 1.  4. 10.]
[31.9 40.7 42.5 47.8 86.6]
{1: 377, 2: 713, 3: 179, 4: 141, 5: 590}


1.0


The last merge heights show one very large gap (from 47.8 to 86.6): at the top, the tree splits the listings into two groups that are far apart. Below that the heights grow slowly, so there is no single natural number of clusters between three and six; five is chosen here to compare with k-means below.

The **adjusted Rand index** (ARI) used above compares two partitions of the same rows: 1 means identical (whatever the label numbers), about 0 means no more agreement than chance.

### In practice

- **Gene-expression heat maps.** Eisen et al. (1998, *PNAS*) introduced the clustered heat map with dendrograms on both axes; it remains the standard figure in genomics.
- **Phylogenetics.** Average linkage (UPGMA) is a classical method for building trees of species from genetic distances.
- **Document and product taxonomies.** Hierarchical clustering of product descriptions or support tickets proposes a first category tree that people then edit; the customs nomenclature used in Sessions 13–16 is such a tree, built by people.

> **Warning.** **Hierarchical clustering does not scale to large n.** It needs all pairwise distances: memory grows with n². Up to about 10,000–20,000 rows it is fine; beyond, cluster a sample or use k-means first.

> **Caution.** **Single linkage chains.** One row lying between two groups can join them. [Extras notebook 05](extras/05-linkage-comparison.ipynb) (scikit-learn gallery) compares the four linkages on six toy datasets.

## Density-based clustering: DBSCAN

### Concept

**DBSCAN** (Density-Based Spatial Clustering of Applications with Noise; Ester et al., 1996) defines clusters as dense regions. It has two parameters: a radius **eps** and a count **min_samples**.

- A **core point** has at least min_samples rows (itself included) within distance eps.
- A **border point** is within eps of a core point but is not core itself.
- A **noise point** is neither; DBSCAN gives it the label −1.

Clusters grow by connecting core points that lie within eps of each other, plus their border points. The number of clusters is a result, not an input.

Worked example: the values 1, 2, 3, 7, 12, 13, 14 with eps = 1.5 and min_samples = 3.

| Value | Rows within 1.5 (incl. itself) | Type |
|---|---|---|
| 2 | 1, 2, 3 | core |
| 1, 3 | two each | border (next to 2) |
| 13 | 12, 13, 14 | core |
| 12, 14 | two each | border (next to 13) |
| 7 | only itself | noise |

Result: two clusters, {1, 2, 3} and {12, 13, 14}, and one noise point, 7.

```mermaid
flowchart LR
    P["A row"] --> C{"At least min_samples<br/>rows within eps?"}
    C -->|yes| CORE["Core point:<br/>starts or extends a cluster"]
    C -->|no| N{"Within eps of<br/>a core point?"}
    N -->|yes| BORDER["Border point:<br/>joins that cluster"]
    N -->|no| NOISE["Noise: label -1"]
```

### Why it matters

DBSCAN finds clusters of any shape (rings, bands) and does not force every row into a cluster. That makes it useful when outliers are expected. Its weakness is one global eps: when clusters differ strongly in density, no single eps fits all. **HDBSCAN** (in scikit-learn since version 1.3) varies the density level and needs only a minimum cluster size ([extras/07-hdbscan.ipynb](extras/07-hdbscan.ipynb)).

### How it works in Python

In [9]:
# the worked example
values = np.array([1, 2, 3, 7, 12, 13, 14], dtype=float).reshape(-1, 1)
print(DBSCAN(eps=1.5, min_samples=3).fit(values).labels_)     # [ 0  0  0 -1  1  1  1]

# on the six standardised features; choose eps from the distance to the 10th neighbour (k-distance)
dist, _ = NearestNeighbors(n_neighbors=10).fit(Z).kneighbors(Z)
print(np.quantile(dist[:, -1], [0.5, 0.9, 0.99]).round(2))      # [0.57 0.91 1.46]
for eps in (0.5, 1.0):
    labels = pd.Series(DBSCAN(eps=eps, min_samples=10).fit(Z).labels_)
    print(eps, labels.nunique() - (labels == -1).any(), "clusters,", (labels == -1).sum(), "noise points")
# 0.5 11 clusters, 4091 noise points
# 1.0 1 clusters, 229 noise points

# on the map: dense areas of listings (coordinates in km, eps = 300 m)
C = d[["east_km", "north_km"]].to_numpy()
dist, _ = NearestNeighbors(n_neighbors=30).fit(C).kneighbors(C)
print(np.quantile(dist[:, -1], [0.1, 0.5, 0.9]).round(2))       # [0.18 0.34 1.6 ]  km to the 30th neighbour
spots = DBSCAN(eps=0.3, min_samples=30).fit(C).labels_
d["hot_spot"] = spots
print(len(set(spots)) - 1, "hot spots,", (spots == -1).sum(), "listings outside")   # 24 hot spots, 3499 listings outside
top = (d[d["hot_spot"] >= 0].groupby("hot_spot")
       .agg(listings=("id", "size"), area=("neighbourhood", lambda s: s.mode()[0]), median_price=("price", "median"))
       .nlargest(3, "listings"))
print(top)
#           listings                      area  median_price
# hot_spot
# 0             1703            Alexanderplatz         170.5
# 1              789      südliche Luisenstadt         129.0
# 2              585  Frankfurter Allee Süd FK         139.8

[ 0  0  0 -1  1  1  1]
[0.57 0.91 1.46]


0.5 11 clusters, 4091 noise points


1.0 1 clusters, 229 noise points
[0.18 0.34 1.6 ]
24 hot spots, 3499 listings outside
          listings                      area  median_price
hot_spot                                                  
0             1703            Alexanderplatz         170.5
1              789      südliche Luisenstadt         129.0
2              585  Frankfurter Allee Süd FK         139.8


On the six features DBSCAN shows what kind of cloud the listings form. With a small eps it finds eleven small, dense groups (mostly listings with identical minimum nights and no recent reviews) and declares half the listings noise; with a larger eps everything joins one cluster, except 229 noise points. There are no dense blobs separated by empty space: the listings form one continuous cloud with denser and sparser regions. This is a finding, not a failure. It means that any segmentation of these listings cuts a continuum, and the 229 noise points at eps = 1.0 are the unusual listings that the lecture of block 2 examines as anomalies.

On the map, DBSCAN does what it was designed for. With a radius of 300 m and at least 30 listings, it finds 24 hot spots of Airbnb listings; the largest, around Alexanderplatz and Mitte, has 1,703 listings, followed by parts of Kreuzberg (südliche Luisenstadt) and Friedrichshain. 3,499 listings lie outside any hot spot; their median distance to Alexanderplatz is 6.5 km, against 3.1 km inside. A city office could use such hot spots to decide where to check registration numbers first.

### In practice

- **Spatial data.** DBSCAN was designed for spatial databases; it is used to find hot spots in GPS points, for example stops in vehicle trajectories or clusters of reported incidents on a map, and, as above, concentrations of short-term rentals.
- **Astronomy.** Density-based methods (including HDBSCAN) are used to find star clusters and stellar streams in the Gaia catalogue of the European Space Agency.
- **Recognition.** The DBSCAN paper received the SIGKDD Test of Time Award in 2014.

> **Warning.** **eps depends on the scale and the number of columns.** Always scale first, and choose eps from the k-distance quantiles (or a k-distance plot: sorted distances, look for the bend), not by guessing.

> **Tip.** In many dimensions distances become similar to each other, and density is hard to estimate. Reduce to a few principal components first (lecture of block 2) when you have more than about ten columns.

> 📖 **Read more:** [extras/07-hdbscan.ipynb](extras/07-hdbscan.ipynb) (scikit-learn gallery): HDBSCAN generalises DBSCAN: it needs no `eps`, finds clusters of different density at the same time, and is less sensitive to its settings `min_cluster_size` and `min_samples`.

## Evaluating and interpreting clusters

### Concept

Without a target, a clustering is judged on three questions.

1. **Separation (internal criteria).** Are rows closer to their own cluster than to others? The mean **silhouette** (see above) compares methods and values of k on the same data. A value above about 0.5 indicates clear structure; values around 0.2 indicate weak, overlapping groups (rules of thumb, Kaufman and Rousseeuw, 1990).
2. **Stability.** Would a slightly different sample give the same clusters? Draw bootstrap samples (rows drawn with replacement), refit, predict the cluster of every original row and compare the partitions with the ARI. Stable clusters have ARI close to 1.
3. **Interpretation (cluster profiles).** A **profile** is a table of means (or medians) of each feature per cluster, ideally on the original scale, plus the size of each cluster and variables not used in the clustering. If no one can describe a cluster in one sentence, it will not be used.

### Why it matters

A segmentation that changes with the random seed, or that only splits a continuous cloud at arbitrary points, should not drive decisions. Reporting silhouette, stability and profiles together lets a reader judge how much structure there really is.

### How it works in Python

In [10]:
km = KMeans(n_clusters=5, n_init=10, random_state=0).fit(Z)
print(round(silhouette_score(Z, km.labels_), 3))                  # 0.29   k-means, all listings
print(round(silhouette_score(Zs, km.labels_[idx]), 3),            # 0.302  k-means on the Ward sample
      round(silhouette_score(Zs, agg.labels_), 3))                # 0.257  Ward
print(round(adjusted_rand_score(km.labels_[idx], agg.labels_), 2))   # 0.65: the methods largely agree

# stability: refit on bootstrap samples, compare with the first fit
aris = []
for seed in range(10):
    boot = rng.choice(len(Z), size=len(Z), replace=True)
    refit = KMeans(n_clusters=5, n_init=10, random_state=seed).fit(Z[boot])
    aris.append(adjusted_rand_score(km.labels_, refit.predict(Z)))
print(np.round([min(aris), np.median(aris)], 2))                   # [0.96 0.98]  very stable

# profile on the original scale, plus variables NOT used in the clustering
d["cluster"] = km.labels_
profile = d.groupby("cluster").agg(
    listings=("id", "size"), price=("price", "median"), guests=("accommodates", "median"),
    min_nights=("minimum_nights", "median"), open_days=("availability_365", "median"),
    reviews_ltm=("number_of_reviews_ltm", "median"), km_centre=("km_centre", "median"),
    entire_home=("room_type", lambda s: (s == "Entire home/apt").mean()),
    multi_host=("calculated_host_listings_count", lambda s: (s > 1).mean()),
    licence=("license", lambda s: s.notna().mean()),             # registration field filled in
)
print(profile.round(2))
#          listings   price  guests  min_nights  open_days  reviews_ltm  km_centre  entire_home  multi_host  licence
# cluster
# 0            2514  153.00     2.0         1.0      307.0         15.0       3.62         0.68        0.70     0.98
# 1            2412  136.55     2.0         2.0       63.0          8.0       4.01         0.64        0.41     0.99
# 2            1698   22.84     2.0        92.0      275.0          0.0       4.33         0.82        0.52     0.20
# 3            1037  293.00     7.0         1.0      273.0         18.0       3.72         0.95        0.81     0.99
# 4             778  119.10     3.0         2.0      257.0          4.0      13.61         0.65        0.46     0.95
outer = ["Spandau", "Marzahn - Hellersdorf", "Treptow - Köpenick", "Steglitz - Zehlendorf", "Reinickendorf"]
print(pd.crosstab(d["district"], d["cluster"], normalize="index").loc[["Mitte", "Neukölln"] + outer].round(2))
# cluster                   0     1     2     3     4
# district
# Mitte                  0.34  0.27  0.23  0.16  0.00
# Neukölln               0.22  0.42  0.18  0.09  0.10
# Spandau                0.00  0.00  0.10  0.01  0.89
# Marzahn - Hellersdorf  0.04  0.09  0.24  0.08  0.55
# Treptow - Köpenick     0.06  0.16  0.15  0.05  0.57
# Steglitz - Zehlendorf  0.05  0.08  0.28  0.06  0.53
# Reinickendorf          0.19  0.14  0.26  0.00  0.41

0.29
0.302 0.257
0.65


[0.96 0.98]
         listings   price  guests  min_nights  open_days  reviews_ltm  km_centre  entire_home  multi_host  licence
cluster                                                                                                           
0            2514  153.00     2.0         1.0      307.0         15.0       3.62         0.68        0.70     0.98
1            2412  136.55     2.0         2.0       63.0          8.0       4.01         0.64        0.41     0.99
2            1698   22.84     2.0        92.0      275.0          0.0       4.33         0.82        0.52     0.20
3            1037  293.00     7.0         1.0      273.0         18.0       3.72         0.95        0.81     0.99
4             778  119.10     3.0         2.0      257.0          4.0      13.61         0.65        0.46     0.95
cluster                   0     1     2     3     4
district                                           
Mitte                  0.34  0.27  0.23  0.16  0.00
Neukölln               0.22

Reading: a silhouette of 0.29 means weak to moderate separation, in line with the continuous cloud that DBSCAN showed. But the partition is very stable under resampling (median ARI 0.98), Ward on a sample largely agrees with it (ARI 0.65), and every cluster can be described in one sentence:

| Cluster | Description | Evidence |
|---|---|---|
| 0 | **Year-round holiday flats**, often run by hosts with several listings | bookable 307 nights, 1-night minimum, 15 reviews a year, 70 % multi-listing hosts |
| 1 | **Occasional lets**, probably people renting out their own home now and then | bookable only 63 nights, 41 % multi-listing hosts |
| 2 | **Medium-term rentals** of three months or more | 92-night minimum, no recent reviews, a nightly price of €23, registration field filled in for only 20 % |
| 3 | **Large group flats** | 7 guests, €293 a night, 95 % entire homes |
| 4 | **Outer districts** | 13.6 km from the centre; 89 % of the Spandau listings |

Three findings deserve a remark. The district table confirms cluster 4 with variables that the clustering did not see: almost no listing in Mitte belongs to it, most listings in Spandau do. And cluster 2 reveals a data problem: a median of €23 a night for entire flats is not a short-stay price. These listings are let for months at a time, and their scraped nightly price is not comparable with the price of a holiday flat. Anyone who models "the price of an Airbnb night" (Sessions 6 and 10) should treat them separately; the lecture of block 2 does so. Finally, the registration field: 95–99 % of the listings in the short-stay clusters show something in it (including placeholders, Session 4), but only 20 % of the medium-term rentals. Whether that reflects the rules for longer lets or missing registrations cannot be decided from the data; it is a question for the city office, not for the algorithm.

A clustering finds whatever dominates the distance, not necessarily what you care about. Here the six columns were chosen to describe the *offer*; adding ten amenity columns would make "similar" mean "similar equipment". The same happens with text: k-means on word counts of descriptions written in several languages groups them mainly by language, because German and French texts share almost no words (Sessions 13 and 14 return to this).

### In practice

- **Market research.** Commercial segmentations are typically validated by checking that segments differ on variables not used to build them (purchase behaviour, survey answers) and that they can be reproduced in a new survey wave.
- **Clinical research.** Data-driven patient subgroups, for example the five adult-onset diabetes clusters of Ahlqvist et al. (2018, *The Lancet Diabetes & Endocrinology*), were replicated in independent cohorts before being discussed as clinically relevant.
- **Housing policy.** Inside Airbnb reports listings by simple rules (entire homes, high availability, hosts with several listings) to estimate how many homes are used for short-term rental rather than housing; a clustering like the one above is a data-driven check of such rules.
- **Benchmarks with known labels.** On data with known classes, the ARI between clusters and classes is used to compare clustering algorithms ([extras/08-clustering-algorithms-compared.ipynb](extras/08-clustering-algorithms-compared.ipynb) shows the visual version).

> **Warning.** **Do not compare silhouettes across different feature sets or scalings.** The silhouette depends on the distance; it is only comparable for different clusterings of the same matrix.

> **Caution.** **Profiles on transformed data hide the meaning.** "Cluster 2 has a mean of −1.2 on standardised availability" means nothing to a city official. Report profiles in terms people know (median price in euros, nights, kilometres, shares of room types and districts) and add the cluster size.

*Worked example:* part A of [03-case-study-airbnb-listing-segments.ipynb](03-case-study-airbnb-listing-segments.ipynb) compares k-means, hierarchical clustering and DBSCAN on the listings, finds the hot spots of listings on the map and describes the clusters by district and room type.

## Check your understanding

1. Why is a clustering harder to evaluate than a classifier? Name three kinds of evidence you can use instead.
2. Customers A, B and C above: recompute the two distances after standardising. Why does the nearest neighbour of A change?
3. Run k-means by hand on the values 0, 1, 5, 6 with k = 2 and starting centres 0 and 1. How many iterations are needed, and what is the final inertia?
4. The inertia falls from k = 5 to k = 6. Does this mean six clusters are better? What would you check?
5. A segment has a churn rate of 47 %. Was churn used to build the segment? Why does that matter for how you report it?
6. Build the single-linkage and complete-linkage trees for the values 0, 3, 4, 10 by hand. Where would you cut to get two clusters?
7. In the DBSCAN example, what happens with eps = 1.5 and min_samples = 4? And with eps = 5 and min_samples = 3?
8. DBSCAN on the six listing features returns one cluster and 229 noise points at eps = 1.0. Is this a failure of DBSCAN? What does it tell you about the listings?
9. The listing clusters have silhouette 0.29 and median bootstrap ARI 0.98. Write two sentences for a report that describe what these numbers mean.
10. Why should a cluster profile include a variable that was not used to build the clusters? Which variables did that job here?
11. The medium-term rentals have a median nightly price of €23. What does this mean for a price model trained on all listings?

## Further reading

- James, G., Witten, D., Hastie, T., Tibshirani, R. and Taylor, J. (2023). *An Introduction to Statistical Learning with Applications in Python*, Chapter 12.4 "Clustering Methods". Springer. https://www.statlearning.com/
- scikit-learn developers (2026). *User Guide: 2.3 Clustering*, sections on k-means, hierarchical clustering, DBSCAN, HDBSCAN and clustering performance evaluation. https://scikit-learn.org/stable/modules/clustering.html
- VanderPlas, J. (2016). *Python Data Science Handbook*, Chapter 5.11 "In Depth: k-Means Clustering". O'Reilly. https://jakevdp.github.io/PythonDataScienceHandbook/05.11-k-means.html
- Rousseeuw, P. J. (1987). Silhouettes: a graphical aid to the interpretation and validation of cluster analysis. *Journal of Computational and Applied Mathematics*, 20, 53–65. https://doi.org/10.1016/0377-0427(87)90125-7
- Ester, M., Kriegel, H.-P., Sander, J. and Xu, X. (1996). A density-based algorithm for discovering clusters in large spatial databases with noise. *Proceedings of KDD-96*, 226–231. https://cdn.aaai.org/KDD/1996/KDD96-037.pdf
- Hennig, C. (2007). Cluster-wise assessment of cluster stability. *Computational Statistics & Data Analysis*, 52(1), 258–271. https://doi.org/10.1016/j.csda.2006.11.025

> 📖 **Read more:** [extras/21-islp-unsupervised-lab.ipynb](extras/21-islp-unsupervised-lab.ipynb) (ISLP labs): the Chapter 12 lab in Python: PCA, matrix completion, k-means and hierarchical clustering on the USArrests data and on the NCI60 gene-expression data; its hierarchical-clustering part compares complete, average and single linkage and cuts the dendrogram of the NCI60 cancer cell lines. Needs the package `ISLP`.